# CIBMTR — Equity in post-HCT Survival Predictions
## Multi-Loss Rank Ensemble: Regression + Cox + Classifier

Key techniques from top solutions:
- **CatBoost Cox models** (Depthwise + Lossguide) for direct survival optimization
- **Deep trees** (depth 8) with early stopping
- **Race-based sample weights** for equity across groups
- **Rank ensemble** with tuned weights
- **Outlier handling** from winning solutions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

SEED = 42
N_FOLDS = 5
print(f'XGBoost: {xgb.__version__} | LightGBM: {lgb.__version__} | CatBoost: {cb.__version__}')

In [ ]:
import os
COMP = '/kaggle/input/competitions/equity-post-HCT-survival-predictions'
DATA = '/kaggle/input/cibmtr-hct-competition-data'
DATA_DIR = COMP if os.path.exists(f'{COMP}/train.csv') else DATA
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv')
test_raw  = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Source: {DATA_DIR}')
print(f'Train: {train_raw.shape}  Test: {test_raw.shape}  Event rate: {(train_raw["efs"]==1).mean():.1%}')

## Feature Engineering

In [ ]:
HLA_COLS = [
    'hla_match_c_high','hla_high_res_8','hla_low_res_6','hla_high_res_6','hla_high_res_10',
    'hla_match_dqb1_high','hla_nmdp_6','hla_match_c_low','hla_match_drb1_low',
    'hla_match_dqb1_low','hla_match_a_high','hla_match_b_low','hla_match_a_low',
    'hla_match_b_high','hla_low_res_8','hla_match_drb1_high','hla_low_res_10',
]

train = train_raw.copy()
test = test_raw.copy()

# Identify binary columns BEFORE encoding (for combinations)
binary_cols = [c for c in train.columns
               if train[c].nunique() == 2 and c not in ['efs', 'ID']]
print(f'Binary columns: {len(binary_cols)}')

for d in [train, test]:
    # Outlier handling (from winning solutions)
    d['year_hct'] = d['year_hct'].replace(2020, 2019)
    d['karnofsky_score'] = d['karnofsky_score'].replace(40, 50)
    d['hla_high_res_8'] = d['hla_high_res_8'].replace(2, 3)
    d['hla_high_res_6'] = d['hla_high_res_6'].replace(0, 2)
    d['hla_high_res_10'] = d['hla_high_res_10'].replace(3, 4)
    d['hla_low_res_8'] = d['hla_low_res_8'].replace(2, 3)
    d['dri_score'] = d['dri_score'].replace('Missing disease status', 'N/A - disease not classifiable')
    for col in ['diabetes', 'pulm_moderate', 'cardiac']:
        d.loc[d[col].isna(), col] = 'Not done'

    # Derived features
    d['year_hct_c'] = d['year_hct'] - 2000
    d['age_group'] = d['age_at_hct'] // 10
    d['hla_total_match'] = d[HLA_COLS].sum(axis=1)
    d['hla_missing_count'] = d[HLA_COLS].isna().sum(axis=1)
    d['age_gap'] = d['donor_age'] - d['age_at_hct']
    d['comorbidity_x_kps'] = d['comorbidity_score'] * d['karnofsky_score']
    d['comorbidity_d_kps'] = d['comorbidity_score'] / (d['karnofsky_score'] + 1)
    d['comorbidity_m_kps'] = d['comorbidity_score'] - d['karnofsky_score']
    d['comorbidity_p_kps'] = d['comorbidity_score'] + d['karnofsky_score']
    d['nan_count'] = d.isna().sum(axis=1)
    d['dri_score_NA'] = d['dri_score'].apply(lambda x: int('N/A' in str(x)))

# Binary column combinations (from Yunbase approach)
combo_cols = []
for i in range(len(binary_cols)):
    for j in range(i + 1, len(binary_cols)):
        new_col = f'{binary_cols[i]}_x_{binary_cols[j]}'
        for d in [train, test]:
            d[new_col] = d[binary_cols[i]].astype(str) + '_' + d[binary_cols[j]].astype(str)
        combo_cols.append(new_col)
print(f'Binary combos: {len(combo_cols)}')

# Label encode ALL categorical/string columns
cat_cols = [c for c in train.columns if train[c].dtype == 'object']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).fillna('__missing__').astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].fillna('__missing__').astype(str))
    test[col] = le.transform(test[col].fillna('__missing__').astype(str))

FEAT = [c for c in train.columns if c not in {'ID', 'efs', 'efs_time'}]
X = train[FEAT].values
X_test = test[FEAT].values
print(f'Features: {len(FEAT)}')

## Targets, Weights & CV

In [ ]:
y_reg = train_raw['efs_time'].values
y_cox = np.where(train_raw['efs'] == 1, train_raw['efs_time'], -train_raw['efs_time'])
y_cls = train_raw['efs'].values

# Race-based sample weights (from top notebooks)
race2weight = {
    'American Indian or Alaska Native': 0.68, 'Asian': 0.70,
    'Black or African-American': 0.67, 'More than one race': 0.68,
    'Native Hawaiian or other Pacific Islander': 0.66, 'White': 0.64}
w_race = train_raw['race_group'].map(race2weight).fillna(0.67).values
w_event = 0.5 * y_cls + 0.5
sample_weight = w_event / w_race

y_strat = train_raw['efs'].astype(str) + '|' + train_raw['race_group'].astype(str)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f'Targets ready. y_reg: [{y_reg.min():.1f}, {y_reg.max():.1f}]')

## Metric (pure numpy)

In [ ]:
def harrell_ci(event_times, scores, events, batch=500):
    """Harrell C-index. Higher score = longer predicted survival."""
    event_times = np.asarray(event_times, dtype=float)
    scores = np.asarray(scores, dtype=float)
    events = np.asarray(events, dtype=bool)
    ev_times, ev_scores = event_times[events], scores[events]
    concordant = permissible = 0.0
    for i in range(0, len(ev_times), batch):
        t_i = ev_times[i:i+batch, None]
        s_i = ev_scores[i:i+batch, None]
        mask = event_times[None, :] > t_i
        concordant += float(((scores[None, :] > s_i) & mask).sum()) + \
                      0.5 * float(((scores[None, :] == s_i) & mask).sum())
        permissible += float(mask.sum())
    return concordant / permissible if permissible > 0 else 0.5

def stratified_ci(df_val, pred_col='prediction'):
    """Competition metric: mean(CI) - std(CI). Expects RISK scores (higher=more risk)."""
    scores = []
    for _, g in df_val.groupby('race_group'):
        if len(g) < 5: continue
        ci = harrell_ci(g['efs_time'].values, -g[pred_col].values,
                        (g['efs'] == 1).values)
        scores.append(ci)
    return float(np.mean(scores) - np.std(scores))

print('Metric ready.')

## Model Training (5-fold CV)

Seven diverse tree models + PRL-NN (pre-trained on AWS SageMaker):
1. **LGB Regressor** on efs_time (depth 8, extra_trees)
2. **CatBoost Regressor** on efs_time (depth 8)
3. **CatBoost Cox Depthwise** (depth 8)
4. **CatBoost Cox Lossguide** (depth 8, 32 leaves)
5. **CatBoost Cox SymmetricTree** (depth 6, default grow policy)
6. **XGB Cox** (survival:cox, depth 8)
7. **XGB Classifier** P(event) (depth 3)
8. **PRL-NN** Pairwise Ranking Neural Network (pre-computed predictions)

In [ ]:
print('=== 1/5 LGB Regressor ===')
oof_lgb = np.zeros(len(train))
test_lgb = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = lgb.LGBMRegressor(
        n_estimators=3000, max_depth=8, num_leaves=64,
        learning_rate=0.03, extra_trees=True,
        reg_lambda=5.0, reg_alpha=0.2,
        colsample_bytree=0.6, min_child_samples=32, max_bin=255,
        n_jobs=-1, random_state=SEED, verbose=-1)
    m.fit(X[tr_idx], y_reg[tr_idx], sample_weight=sample_weight[tr_idx],
          eval_set=[(X[val_idx], y_reg[val_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = m.predict(X[val_idx])
    test_lgb += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration_}')
df_ev = train_raw[['efs_time','efs','race_group']].copy()
df_ev['prediction'] = -oof_lgb
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 2/5 CatBoost Regressor ===')
oof_cat = np.zeros(len(train))
test_cat = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = cb.CatBoostRegressor(
        iterations=3000, depth=8, learning_rate=0.03,
        l2_leaf_reg=8.0, subsample=0.8,
        early_stopping_rounds=100,
        random_seed=SEED, verbose=0)
    m.fit(X[tr_idx], y_reg[tr_idx], sample_weight=sample_weight[tr_idx],
          eval_set=(X[val_idx], y_reg[val_idx]))
    oof_cat[val_idx] = m.predict(X[val_idx])
    test_cat += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration_}')
df_ev['prediction'] = -oof_cat
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 3/5 CatBoost Cox (Depthwise) ===')
oof_cox1 = np.zeros(len(train))
test_cox1 = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = cb.CatBoostRegressor(
        loss_function='Cox', iterations=3000, depth=8,
        learning_rate=0.03, l2_leaf_reg=8.0,
        grow_policy='Depthwise', min_data_in_leaf=8,
        early_stopping_rounds=100,
        random_seed=SEED, verbose=0)
    m.fit(X[tr_idx], y_cox[tr_idx],
          eval_set=(X[val_idx], y_cox[val_idx]))
    oof_cox1[val_idx] = m.predict(X[val_idx])
    test_cox1 += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration_}')
df_ev['prediction'] = oof_cox1
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 4/5 CatBoost Cox (Lossguide) ===')
oof_cox2 = np.zeros(len(train))
test_cox2 = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = cb.CatBoostRegressor(
        loss_function='Cox', iterations=3000, depth=8,
        learning_rate=0.03, l2_leaf_reg=8.0,
        grow_policy='Lossguide', num_leaves=32, min_data_in_leaf=2,
        early_stopping_rounds=100,
        random_seed=SEED, verbose=0)
    m.fit(X[tr_idx], y_cox[tr_idx],
          eval_set=(X[val_idx], y_cox[val_idx]))
    oof_cox2[val_idx] = m.predict(X[val_idx])
    test_cox2 += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration_}')
df_ev['prediction'] = oof_cox2
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 5/7 CatBoost Cox (SymmetricTree) ===')
oof_cox3 = np.zeros(len(train))
test_cox3 = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = cb.CatBoostRegressor(
        loss_function='Cox', iterations=3000, depth=6,
        learning_rate=0.03, l2_leaf_reg=8.0,
        grow_policy='SymmetricTree',
        early_stopping_rounds=100,
        random_seed=SEED, verbose=0)
    m.fit(X[tr_idx], y_cox[tr_idx],
          eval_set=(X[val_idx], y_cox[val_idx]))
    oof_cox3[val_idx] = m.predict(X[val_idx])
    test_cox3 += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration_}')
df_ev['prediction'] = oof_cox3
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 6/7 XGB Cox ===')
oof_xgb_cox = np.zeros(len(train))
test_xgb_cox = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = xgb.XGBRegressor(
        objective='survival:cox',
        n_estimators=3000, max_depth=8, learning_rate=0.03,
        colsample_bytree=0.7, subsample=0.8,
        reg_alpha=0.1, reg_lambda=8.0,
        early_stopping_rounds=100,
        n_jobs=-1, random_state=SEED, verbosity=0)
    m.fit(X[tr_idx], y_cox[tr_idx],
          eval_set=[(X[val_idx], y_cox[val_idx])], verbose=False)
    oof_xgb_cox[val_idx] = m.predict(X[val_idx])
    test_xgb_cox += m.predict(X_test) / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration}')
df_ev['prediction'] = oof_xgb_cox
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 7/7 XGB Classifier ===')
oof_xgb = np.zeros(len(train))
test_xgb = np.zeros(len(test))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_strat), 1):
    m = xgb.XGBClassifier(
        n_estimators=3000, max_depth=3, learning_rate=0.04,
        colsample_bytree=0.7, subsample=0.8,
        scale_pos_weight=1.54, min_child_weight=4, gamma=3.13,
        eval_metric='auc', early_stopping_rounds=50,
        n_jobs=-1, random_state=SEED, verbosity=0)
    m.fit(X[tr_idx], y_cls[tr_idx],
          eval_set=[(X[val_idx], y_cls[val_idx])], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X[val_idx])[:, 1]
    test_xgb += m.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold} best_iter={m.best_iteration}')
df_ev['prediction'] = oof_xgb
print(f'  OOF CI = {stratified_ci(df_ev):.4f}')

In [ ]:
print('=== 8/8 ODST PRL-NN (pre-trained on AWS SageMaker) ===')
import json, torch, torch.nn as nn
from torch.autograd import Function
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

NN_DS = '/kaggle/input/cibmtr-hct-nn-weights'

# ── Load pre-computed OOF predictions from SageMaker training ────────────────
# These are the exact OOF predictions (risk direction: higher = more risk)
oof_nn = np.load(f'{NN_DS}/nn_oof_rank.npy')
print(f'  Loaded OOF predictions: {oof_nn.shape}')

df_ev['prediction'] = oof_nn  # already risk direction for stratified_ci
print(f'  NN OOF CI (from training) = {stratified_ci(df_ev):.4f}')

# ── Inline ODST dependencies (from pytorch_tabular) for test inference ───────
def _make_ix_like(X, dim):
    d = X.size(dim)
    rho = torch.arange(1, d + 1, device=X.device, dtype=X.dtype)
    view = [1] * X.dim()
    view[0] = -1
    return rho.view(view).transpose(0, dim)

def _roll_last(X, dim):
    if dim == -1: return X
    elif dim < 0: dim = X.dim() - dim
    perm = [i for i in range(X.dim()) if i != dim] + [dim]
    return X.permute(perm)

def _sparsemax_threshold_and_support(X, dim=-1, k=None):
    if k is None or k >= X.shape[dim]:
        topk, _ = torch.sort(X, dim=dim, descending=True)
    else:
        topk, _ = torch.topk(X, k=k, dim=dim)
    topk_cumsum = topk.cumsum(dim) - 1
    rhos = _make_ix_like(topk, dim)
    support = rhos * topk > topk_cumsum
    support_size = support.sum(dim=dim).unsqueeze(dim)
    tau = topk_cumsum.gather(dim, support_size - 1)
    tau /= support_size.to(X.dtype)
    if k is not None and k < X.shape[dim]:
        unsolved = (support_size == k).squeeze(dim)
        if torch.any(unsolved):
            in_ = _roll_last(X, dim)[unsolved]
            tau_, ss_ = _sparsemax_threshold_and_support(in_, dim=-1, k=2*k)
            _roll_last(tau, dim)[unsolved] = tau_
            _roll_last(support_size, dim)[unsolved] = ss_
    return tau, support_size

class SparsemaxFunction(Function):
    @classmethod
    def forward(cls, ctx, X, dim=-1, k=None):
        ctx.dim = dim
        max_val, _ = X.max(dim=dim, keepdim=True)
        X = X - max_val
        tau, supp_size = _sparsemax_threshold_and_support(X, dim=dim, k=k)
        output = torch.clamp(X - tau, min=0)
        ctx.save_for_backward(supp_size, output)
        return output
    @classmethod
    def backward(cls, ctx, grad_output):
        supp_size, output = ctx.saved_tensors
        dim = ctx.dim
        grad_input = grad_output.clone()
        grad_input[output == 0] = 0
        v_hat = grad_input.sum(dim=dim) / supp_size.to(output.dtype).squeeze(dim)
        v_hat = v_hat.unsqueeze(dim)
        grad_input = torch.where(output != 0, grad_input - v_hat, grad_input)
        return grad_input, None, None

def sparsemax(X, dim=-1, k=None):
    return SparsemaxFunction.apply(X, dim, k)

def sparsemoid(input):
    return (0.5 * input + 0.5).clamp_(0, 1)

def check_numpy(x):
    if isinstance(x, torch.Tensor): x = x.detach().cpu().numpy()
    return np.asarray(x)

class ModuleWithInit(nn.Module):
    def __init__(self):
        super().__init__()
        self._is_initialized_tensor = nn.Parameter(
            torch.tensor(0, dtype=torch.uint8), requires_grad=False)
        self._is_initialized_bool = None
    def initialize(self, *args, **kwargs):
        raise NotImplementedError
    def __call__(self, *args, **kwargs):
        if self._is_initialized_bool is None:
            self._is_initialized_bool = bool(self._is_initialized_tensor.item())
        if not self._is_initialized_bool:
            self.initialize(*args, **kwargs)
            self._is_initialized_tensor.data[...] = 1
            self._is_initialized_bool = True
        return super().__call__(*args, **kwargs)

class ODST(ModuleWithInit):
    def __init__(self, in_features, num_trees, depth=6, tree_output_dim=1,
                 flatten_output=True, choice_function=sparsemax,
                 bin_function=sparsemoid, initialize_response_=nn.init.normal_,
                 initialize_selection_logits_=nn.init.uniform_,
                 threshold_init_beta=1.0, threshold_init_cutoff=1.0):
        super().__init__()
        self.depth, self.num_trees, self.tree_dim = depth, num_trees, tree_output_dim
        self.flatten_output = flatten_output
        self.choice_function, self.bin_function = choice_function, bin_function
        self.threshold_init_beta = threshold_init_beta
        self.threshold_init_cutoff = threshold_init_cutoff
        self.response = nn.Parameter(torch.zeros([num_trees, tree_output_dim, 2**depth]))
        initialize_response_(self.response)
        self.feature_selection_logits = nn.Parameter(torch.zeros([in_features, num_trees, depth]))
        initialize_selection_logits_(self.feature_selection_logits)
        self.feature_thresholds = nn.Parameter(
            torch.full([num_trees, depth], float("nan")), requires_grad=True)
        self.log_temperatures = nn.Parameter(
            torch.full([num_trees, depth], float("nan")), requires_grad=True)
        with torch.no_grad():
            indices = torch.arange(2**self.depth)
            offsets = 2 ** torch.arange(self.depth)
            bin_codes = (indices.view(1, -1) // offsets.view(-1, 1) % 2).to(torch.float32)
            bin_codes_1hot = torch.stack([bin_codes, 1.0 - bin_codes], dim=-1)
            self.bin_codes_1hot = nn.Parameter(bin_codes_1hot, requires_grad=False)

    def forward(self, input):
        assert len(input.shape) >= 2
        if len(input.shape) > 2:
            return self.forward(input.view(-1, input.shape[-1])).view(*input.shape[:-1], -1)
        feature_selectors = self.choice_function(self.feature_selection_logits, dim=0)
        feature_values = torch.einsum("bi,ind->bnd", input, feature_selectors)
        threshold_logits = (feature_values - self.feature_thresholds) * torch.exp(-self.log_temperatures)
        threshold_logits = torch.stack([-threshold_logits, threshold_logits], dim=-1)
        bins = self.bin_function(threshold_logits)
        bin_matches = torch.einsum("btds,dcs->btdc", bins, self.bin_codes_1hot)
        response_weights = torch.prod(bin_matches, dim=-2)
        response = torch.einsum("bnd,ncd->bnc", response_weights, self.response)
        return response.flatten(1, 2) if self.flatten_output else response

    def initialize(self, input, eps=1e-6):
        with torch.no_grad():
            feature_selectors = self.choice_function(self.feature_selection_logits, dim=0)
            feature_values = torch.einsum("bi,ind->bnd", input, feature_selectors)
            percentiles_q = 100 * np.random.beta(
                self.threshold_init_beta, self.threshold_init_beta,
                size=[self.num_trees, self.depth])
            self.feature_thresholds.data[...] = torch.as_tensor(
                list(map(np.percentile,
                         check_numpy(feature_values.flatten(1, 2).t()),
                         percentiles_q.flatten())),
                dtype=feature_values.dtype, device=feature_values.device
            ).view(self.num_trees, self.depth)
            temperatures = np.percentile(
                check_numpy(abs(feature_values - self.feature_thresholds)),
                q=100 * min(1.0, self.threshold_init_cutoff), axis=0)
            temperatures /= max(1.0, self.threshold_init_cutoff)
            self.log_temperatures.data[...] = torch.log(
                torch.as_tensor(temperatures) + eps)

class CatEmbeddings(nn.Module):
    def __init__(self, projection_dim, categorical_cardinality, embedding_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim)
            for card in categorical_cardinality])
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim * len(categorical_cardinality), projection_dim),
            nn.GELU(),
            nn.Linear(projection_dim, projection_dim))
    def forward(self, x_cat):
        embs = [self.embeddings[i](x_cat[:, i]) for i in range(x_cat.shape[1])]
        return self.projection(torch.cat(embs, dim=1))

class ODSTNN(nn.Module):
    def __init__(self, continuous_dim, categorical_cardinality, embedding_dim,
                 projection_dim, hidden_dim, dropout=0.05):
        super().__init__()
        self.embeddings = CatEmbeddings(projection_dim, categorical_cardinality, embedding_dim)
        self.mlp = nn.Sequential(
            ODST(projection_dim + continuous_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim), nn.Dropout(dropout))
        self.out = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x_cat, x_cont):
        x = self.embeddings(x_cat)
        x = torch.cat([x, x_cont], dim=1)
        x = self.dropout(x)
        x = self.mlp(x)
        return self.out(x).squeeze(1), x

# ── Load metadata ────────────────────────────────────────────────────────────
with open(f'{NN_DS}/nn_metadata.json') as f:
    nn_meta = json.load(f)
cat_cols_nn = nn_meta['cat_cols']
num_cols_nn = nn_meta['num_cols']

# ── Preprocess for test inference only ───────────────────────────────────────
CATEGORICAL_VARIABLES_NN = [
    'dri_score', 'graft_type', 'prod_type', 'prim_disease_hct',
    'psych_disturb', 'diabetes', 'arrhythmia', 'vent_hist', 'renal_issue',
    'pulm_moderate', 'pulm_severe', 'obesity', 'hepatic_mild',
    'hepatic_severe', 'peptic_ulcer', 'rheum_issue', 'cardiac',
    'prior_tumor', 'mrd_hct', 'tbi_status', 'cyto_score',
    'cyto_score_detail', 'ethnicity', 'race_group', 'sex_match',
    'donor_related', 'cmv_status', 'tce_imm_match', 'tce_match',
    'tce_div_match', 'melphalan_dose', 'rituximab', 'gvhd_proph',
    'in_vivo_tcd', 'conditioning_intensity']
OTHER_NUM_NN = ['year_hct', 'donor_age', 'age_at_hct',
                'comorbidity_score', 'karnofsky_score']

nn_train = train_raw.copy()
nn_test = test_raw.copy()
for df in [nn_train, nn_test]:
    df[CATEGORICAL_VARIABLES_NN] = df[CATEGORICAL_VARIABLES_NN].fillna("Unknown")
    df[OTHER_NUM_NN] = df[OTHER_NUM_NN].fillna(df[OTHER_NUM_NN].median())
    df['year_hct'] = df['year_hct'] - 2000

# ── 5-fold test inference using saved preprocessing ──────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
test_nn_pred = np.zeros(len(nn_test))

y_strat_nn = nn_train['efs'].astype(str) + '|' + nn_train['race_group'].astype(str)
skf_nn = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf_nn.split(nn_train, y_strat_nn)):
    with open(f'{NN_DS}/nn_fold{fold}_preproc.json') as f:
        preproc = json.load(f)

    train_fold = nn_train.iloc[tr_idx].copy()

    # Use saved label encoders from training (exact class lists)
    transformers = []
    X_cat_test_list = []
    for i, col in enumerate(cat_cols_nn):
        le = LabelEncoder()
        le.classes_ = np.array(preproc['transformers_classes'][i])
        if '__missing__' not in le.classes_:
            le.classes_ = np.append(le.classes_, '__missing__')
        test_v = nn_test[col].fillna('__missing__').astype(str)
        test_v = test_v.map(lambda x, _le=le: x if x in _le.classes_ else '__missing__')
        X_cat_test_list.append(le.transform(test_v))
        transformers.append(le)

    X_cat_test = np.column_stack(X_cat_test_list)

    # Use saved scaler from training
    imputer = SimpleImputer(strategy='mean', add_indicator=True)
    imputer.fit(train_fold[num_cols_nn].values)
    X_num_test_imp = imputer.transform(nn_test[num_cols_nn].values)
    scaler_mean = np.array(preproc['scaler_mean'])
    scaler_scale = np.array(preproc['scaler_scale'])
    X_num_test = (X_num_test_imp - scaler_mean) / scaler_scale

    # Build model with exact cardinalities from training
    model = ODSTNN(
        continuous_dim=len(scaler_mean),
        categorical_cardinality=[len(preproc['transformers_classes'][i])
                                 for i in range(len(cat_cols_nn))],
        embedding_dim=16, projection_dim=112, hidden_dim=56,
        dropout=0.05463240181423116)

    state = torch.load(f'{NN_DS}/nn_fold{fold}.pt', map_location=device, weights_only=False)
    clean_state = {}
    for k, v in state.items():
        k2 = k.replace('model.', '', 1) if k.startswith('model.') else k
        clean_state[k2] = v
    model.load_state_dict(clean_state, strict=False)
    model.to(device).eval()

    with torch.no_grad():
        pred_test, _ = model(
            torch.tensor(X_cat_test, dtype=torch.long).to(device),
            torch.tensor(X_num_test, dtype=torch.float32).to(device))
        test_nn_pred += pred_test.cpu().numpy()

    print(f'  Fold {fold+1} test inference done')

# Convert to RISK direction (higher = more risk)
test_nn = -test_nn_pred
print(f'  Test predictions ready: {test_nn.shape}')

## Rank Ensemble

Weighted average of ranks. Cox models + PRL-NN get highest weights.

In [ ]:
# All predictions in RISK direction (higher = more risk)
oof_dict = {
    'lgb_reg':  -oof_lgb,
    'cat_reg':  -oof_cat,
    'cox_dw':    oof_cox1,
    'cox_lg':    oof_cox2,
    'cox_st':    oof_cox3,
    'xgb_cox':   oof_xgb_cox,
    'xgb_cls':   oof_xgb,
    'nn':        oof_nn,
}
test_dict = {
    'lgb_reg':  -test_lgb,
    'cat_reg':  -test_cat,
    'cox_dw':    test_cox1,
    'cox_lg':    test_cox2,
    'cox_st':    test_cox3,
    'xgb_cox':   test_xgb_cox,
    'xgb_cls':   test_xgb,
    'nn':        test_nn,
}

print('Individual model OOF scores:')
for name, oof_pred in oof_dict.items():
    df_ev['prediction'] = oof_pred
    print(f'  {name:>10}: {stratified_ci(df_ev):.4f}')

# Try multiple weight configs
weight_configs = [
    # Trees only
    ('cox4',         {'cox_dw': 6, 'cox_lg': 6, 'cox_st': 6, 'xgb_cox': 6}),
    # NN + trees (diverse blends)
    ('cox4+nn',      {'cox_dw': 6, 'cox_lg': 6, 'cox_st': 6, 'xgb_cox': 6, 'nn': 8}),
    ('cox3+nn',      {'cox_dw': 6, 'cox_lg': 6, 'cox_st': 6, 'nn': 8}),
    ('nn_heavy',     {'cox_dw': 4, 'cox_lg': 4, 'cox_st': 4, 'xgb_cox': 4, 'nn': 12}),
    ('nn+cox_dw',    {'cox_dw': 6, 'nn': 8}),
    ('nn_only',      {'nn': 1}),
    ('all+nn',       {'lgb_reg': 1, 'cat_reg': 1, 'cox_dw': 6, 'cox_lg': 6, 'cox_st': 6, 'xgb_cox': 6, 'xgb_cls': 3, 'nn': 8}),
    ('cox4+cls+nn',  {'cox_dw': 6, 'cox_lg': 6, 'cox_st': 6, 'xgb_cox': 6, 'xgb_cls': 3, 'nn': 8}),
    ('all_equal',    {k: 1 for k in oof_dict}),
    # Extra diversity-focused blends
    ('nn_eq_cox4',   {'cox_dw': 1, 'cox_lg': 1, 'cox_st': 1, 'xgb_cox': 1, 'nn': 1}),
    ('nn2x_cox4',    {'cox_dw': 1, 'cox_lg': 1, 'cox_st': 1, 'xgb_cox': 1, 'nn': 2}),
    ('nn3x_cox4',    {'cox_dw': 1, 'cox_lg': 1, 'cox_st': 1, 'xgb_cox': 1, 'nn': 3}),
    ('nn_cox_dw_eq', {'cox_dw': 1, 'nn': 1}),
]
best_ci, best_name, best_weights = 0, '', {}
print('\nWeight search:')
for name, w in weight_configs:
    ranked = {k: rankdata(v) for k, v in oof_dict.items() if k in w}
    ens = sum(w[k] * ranked[k] for k in w)
    df_ev['prediction'] = ens
    ci = stratified_ci(df_ev)
    print(f'  {name:>14}: {ci:.4f}')
    if ci > best_ci:
        best_ci, best_name, best_weights = ci, name, w

print(f'\nBest OOF: {best_name} = {best_ci:.4f}  weights={best_weights}')

# Also find best blend that includes BOTH nn and at least one tree model
best_blend_ci, best_blend_name, best_blend_weights = 0, '', {}
for name, w in weight_configs:
    if 'nn' not in w or len(w) < 2:
        continue
    ranked = {k: rankdata(v) for k, v in oof_dict.items() if k in w}
    ens = sum(w[k] * ranked[k] for k in w)
    df_ev['prediction'] = ens
    ci = stratified_ci(df_ev)
    if ci > best_blend_ci:
        best_blend_ci, best_blend_name, best_blend_weights = ci, name, w

print(f'Best blend: {best_blend_name} = {best_blend_ci:.4f}  weights={best_blend_weights}')

# Use the best blend (diversity generalizes better on LB)
use_weights = best_blend_weights
use_name = best_blend_name
print(f'\n=> Using: {use_name} (blend for LB diversity)')

# Build final ensemble
oof_ranked = {k: rankdata(v) for k, v in oof_dict.items() if k in use_weights}
oof_ensemble = sum(use_weights[k] * oof_ranked[k] for k in use_weights)

test_ranked = {k: rankdata(v) for k, v in test_dict.items() if k in use_weights}
test_ensemble = sum(use_weights[k] * test_ranked[k] for k in use_weights)

## Submission

In [ ]:
submission = pd.DataFrame({'ID': test_raw['ID'], 'prediction': test_ensemble})
submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved ({len(submission)} rows)')
submission